# GPU Performance with PyTorch

This notebook demonstrates:
1. **System inspection** — CPU, RAM, and GPU specs
2. **CPU vs GPU benchmarks** — matrix ops, element-wise ops, neural network training
3. **Scaling behaviour** — how the GPU advantage grows with problem size
4. **Memory transfer overhead** — the hidden cost of moving data to/from the GPU
5. **Practical tips** — when GPU helps and when it doesn't

In [ ]:
!pip install -q torch psutil

---
## 1. System Information

In [ ]:
import torch, sys, psutil, platform

print("-" * 60, "SYSTEM INFORMATION", "-" * 60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Platform:        {platform.system()} {platform.release()}")

print()
print("-" * 60, "CPU INFORMATION", "-" * 60)
print(f"Processor:      {platform.processor()}")
print(f"Physical Cores: {psutil.cpu_count(logical=False)}")
print(f"Logical Cores:  {psutil.cpu_count(logical=True)}")
print(f"CPU Usage:      {psutil.cpu_percent(interval=1)}%")

print()
print("-" * 60, "MEMORY INFORMATION", "-" * 60)
mem = psutil.virtual_memory()
print(f"Total RAM:     {mem.total    / 1024**3:.2f} GB")
print(f"Available RAM: {mem.available / 1024**3:.2f} GB")
print(f"Used RAM:      {mem.used      / 1024**3:.2f} GB")
print(f"Memory Usage:  {mem.percent:.1f}%")

print()
print("-" * 60, "GPU INFORMATION", "-" * 60)

if torch.cuda.is_available():
    print("CUDA Available: Yes")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print()
    for device_id in range(torch.cuda.device_count()):
        device_capability  = torch.cuda.get_device_capability(device_id)
        device_name        = torch.cuda.get_device_name(device_id)
        device_properties  = torch.cuda.get_device_properties(device_id)
        device_memory      = device_properties.total_memory

        print(f"GPU #{device_id + 1}:")
        print(f"  Name:               {device_name}")
        print(f"  Compute Capability: {device_capability[0]}.{device_capability[1]}")
        print(f"  Total Memory:       {device_memory / 1024**3:.2f} GB")
        print(f"  Multiprocessors:    {device_properties.multi_processor_count}")
        print(f"  Max Threads/SM:     {device_properties.max_threads_per_multi_processor}")

        # Memory usage
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated(device_id) / 1024**3
        reserved  = torch.cuda.memory_reserved(device_id)  / 1024**3
        print(f"  Memory Allocated:   {allocated:.2f} GB")
        print(f"  Memory Reserved:    {reserved:.2f} GB")
        print()
else:
    print("CUDA Available: No")
    print("GPU acceleration not available")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('ggplot')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cpu':
    print("\nNote: no GPU detected — all 'GPU' timings below are CPU timings.")
    print("The patterns still illustrate when vectorised ops help.")

---
## 2. Matrix Multiplication: CPU vs GPU

Matrix multiplication (matmul) is the core operation in neural networks — every linear layer, every attention score is a matmul.  
It is also the operation GPUs are most optimised for.

In [ ]:
def benchmark(fn, n_runs=20, warmup=5):
    """Run fn() n_runs times, discard warmup, return mean time in ms."""
    for _ in range(warmup):
        fn()
    if device.type == 'cuda':
        torch.cuda.synchronize()   # wait for all GPU ops to finish

    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        fn()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return np.mean(times), np.std(times)


# Single size comparison first
N = 2048
A_cpu = torch.randn(N, N)
B_cpu = torch.randn(N, N)
A_gpu = A_cpu.to(device)
B_gpu = B_cpu.to(device)

cpu_mean, cpu_std = benchmark(lambda: torch.matmul(A_cpu, B_cpu))
gpu_mean, gpu_std = benchmark(lambda: torch.matmul(A_gpu, B_gpu))

print(f"Matrix multiplication  ({N}×{N} @ {N}×{N})")
print(f"  CPU:     {cpu_mean:8.2f} ms  ± {cpu_std:.2f}")
print(f"  GPU:     {gpu_mean:8.2f} ms  ± {gpu_std:.2f}")
print(f"  Speedup: {cpu_mean/gpu_mean:.1f}×")

In [ ]:
# How does the speedup change with matrix size?
sizes = [64, 128, 256, 512, 1024, 2048, 4096]
cpu_times, gpu_times, speedups = [], [], []

for n in sizes:
    A = torch.randn(n, n)
    B = torch.randn(n, n)
    Ag = A.to(device)
    Bg = B.to(device)

    c, _ = benchmark(lambda: torch.matmul(A, B), n_runs=10)
    g, _ = benchmark(lambda: torch.matmul(Ag, Bg), n_runs=10)

    cpu_times.append(c)
    gpu_times.append(g)
    speedups.append(c / g)
    print(f"  n={n:5d}  CPU={c:8.2f}ms  GPU={g:8.2f}ms  speedup={c/g:6.1f}×")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(sizes, cpu_times, 'o-', label='CPU', color='#3498db')
ax1.plot(sizes, gpu_times, 's-', label=str(device).upper(), color='#e74c3c')
ax1.set_xlabel('Matrix size N  (N×N)')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Matmul Time vs Matrix Size')
ax1.set_yscale('log')
ax1.legend()

ax2.plot(sizes, speedups, 'D-', color='#2ecc71')
ax2.axhline(y=1, color='gray', linestyle='--', label='No speedup')
ax2.set_xlabel('Matrix size N  (N×N)')
ax2.set_ylabel('Speedup (CPU time / GPU time)')
ax2.set_title('GPU Speedup vs Matrix Size')
ax2.legend()

plt.suptitle('Matrix Multiplication: CPU vs GPU', fontsize=13)
plt.tight_layout()
plt.show()
print("Key insight: GPU advantage grows with problem size — small matrices are not worth offloading.")

---
## 3. Operation Comparison: What Benefits Most?

Not all operations benefit equally from a GPU.  
Let's measure speedup for the operations most common in deep learning.

In [ ]:
N = 4096
x_cpu = torch.randn(N, N)
x_gpu = x_cpu.to(device)

operations = [
    # (label, cpu_fn, gpu_fn)
    ('Element-wise add',     lambda: x_cpu + x_cpu,             lambda: x_gpu + x_gpu),
    ('Element-wise mul',     lambda: x_cpu * x_cpu,             lambda: x_gpu * x_gpu),
    ('ReLU',                 lambda: torch.relu(x_cpu),         lambda: torch.relu(x_gpu)),
    ('Sigmoid',              lambda: torch.sigmoid(x_cpu),      lambda: torch.sigmoid(x_gpu)),
    ('Softmax',              lambda: torch.softmax(x_cpu, -1),  lambda: torch.softmax(x_gpu, -1)),
    ('Matrix multiply',      lambda: x_cpu @ x_cpu,             lambda: x_gpu @ x_gpu),
    ('Sum reduction',        lambda: x_cpu.sum(),               lambda: x_gpu.sum()),
    ('Sort',                 lambda: x_cpu.sort(),              lambda: x_gpu.sort()),
    ('Batch norm',           lambda: torch.nn.functional.batch_norm(
                                        x_cpu, None, None, training=True),
                             lambda: torch.nn.functional.batch_norm(
                                        x_gpu, None, None, training=True)),
]

print(f"Benchmarking on {N}×{N} tensors:\n")
print(f"  {'Operation':22s}  {'CPU (ms)':>10s}  {'GPU (ms)':>10s}  {'Speedup':>8s}")
print("  " + "-" * 58)

op_results = []
for label, cpu_fn, gpu_fn in operations:
    try:
        c, _ = benchmark(cpu_fn, n_runs=15)
        g, _ = benchmark(gpu_fn, n_runs=15)
        sp = c / g
        op_results.append((label, c, g, sp))
        print(f"  {label:22s}  {c:10.2f}  {g:10.2f}  {sp:7.1f}×")
    except Exception as e:
        print(f"  {label:22s}  ERROR: {e}")

# Bar chart of speedups
labels_op   = [r[0] for r in op_results]
speedups_op = [r[3] for r in op_results]
colors_op   = ['#2ecc71' if s > 1 else '#e74c3c' for s in speedups_op]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels_op, speedups_op, color=colors_op, alpha=0.85)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1, label='No speedup')
ax.set_xlabel('Speedup (CPU / GPU time)')
ax.set_title(f'GPU Speedup by Operation ({N}×{N} tensors)')
ax.legend()
for i, sp in enumerate(speedups_op):
    ax.text(sp + 0.1, i, f'{sp:.1f}×', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. The Hidden Cost: CPU ↔ GPU Memory Transfer

Moving data between CPU RAM and GPU VRAM takes time.  
For small operations this overhead can **exceed** the compute savings — a common mistake in real code.

In [ ]:
sizes_mb = [0.1, 1, 10, 100, 500]   # tensor sizes in MB

print(f"{'Size':>8s}  {'H→D (ms)':>10s}  {'D→H (ms)':>10s}  {'Bandwidth H→D':>15s}")
print("-" * 52)

transfer_results = []
for mb in sizes_mb:
    n_elems = int(mb * 1024**2 / 4)   # float32 = 4 bytes
    t_cpu = torch.randn(n_elems)

    # Host → Device
    def h2d():
        t = t_cpu.to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        return t

    t_gpu = h2d()

    # Device → Host
    def d2h():
        t = t_gpu.cpu()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        return t

    h2d_ms, _ = benchmark(h2d, n_runs=20)
    d2h_ms, _ = benchmark(d2h, n_runs=20)
    bw_gbs = mb / 1024 / (h2d_ms / 1000)  # GB/s

    transfer_results.append((mb, h2d_ms, d2h_ms, bw_gbs))
    print(f"{mb:>7.1f}MB  {h2d_ms:>10.2f}  {d2h_ms:>10.2f}  {bw_gbs:>12.2f} GB/s")

In [ ]:
# Break-even analysis: when does GPU compute savings overcome transfer cost?
# We measure: pure GPU compute  vs  transfer + GPU compute  vs  CPU compute

print("Break-even: transfer + GPU compute vs pure CPU compute\n")
print(f"  {'Size N':>8s}  {'CPU (ms)':>10s}  {'GPU only (ms)':>14s}  {'Xfer+GPU (ms)':>14s}  {'Winner':>8s}")
print("  " + "-" * 62)

breakeven_results = []
for n in [64, 256, 512, 1024, 2048]:
    A_c = torch.randn(n, n)
    A_g = A_c.to(device)

    cpu_t, _  = benchmark(lambda: A_c @ A_c, n_runs=20)
    gpu_t, _  = benchmark(lambda: A_g @ A_g, n_runs=20)

    def full_pipeline():
        g = A_c.to(device)
        result = g @ g
        return result.cpu()

    xfer_gpu_t, _ = benchmark(full_pipeline, n_runs=20)

    winner = 'GPU' if xfer_gpu_t < cpu_t else 'CPU'
    breakeven_results.append((n, cpu_t, gpu_t, xfer_gpu_t, winner))
    print(f"  {n:>8d}  {cpu_t:>10.2f}  {gpu_t:>14.2f}  {xfer_gpu_t:>14.2f}  {winner:>8s}")

print("\nKey insight: below the break-even size, the transfer dominates — stay on CPU.")

---
## 5. Neural Network Training: CPU vs GPU

A realistic benchmark — training a small MLP for one epoch on synthetic data.

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_one_epoch(model, X, y, batch_size=256):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    n = X.shape[0]
    total_loss = 0
    for i in range(0, n, batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss


# Synthetic dataset: 50K samples, 512 features, 10 classes
N_SAMPLES, IN_DIM, OUT_DIM = 50_000, 512, 10
X_data = torch.randn(N_SAMPLES, IN_DIM)
y_data = torch.randint(0, OUT_DIM, (N_SAMPLES,))

CONFIGS = [
    ('Small MLP',  256,  3),
    ('Medium MLP', 1024, 6),
    ('Large MLP',  2048, 10),
]

print(f"{'Model':15s}  {'CPU (ms/epoch)':>15s}  {'GPU (ms/epoch)':>15s}  {'Speedup':>8s}")
print("-" * 60)

nn_results = []
for name, hidden, layers in CONFIGS:
    # CPU
    model_cpu = MLP(IN_DIM, hidden, OUT_DIM, layers)
    Xc, yc = X_data.clone(), y_data.clone()
    cpu_t, _ = benchmark(lambda: train_one_epoch(model_cpu, Xc, yc), n_runs=3, warmup=1)

    # GPU
    model_gpu = MLP(IN_DIM, hidden, OUT_DIM, layers).to(device)
    Xg, yg = X_data.to(device), y_data.to(device)
    gpu_t, _ = benchmark(lambda: train_one_epoch(model_gpu, Xg, yg), n_runs=3, warmup=1)

    sp = cpu_t / gpu_t
    nn_results.append((name, cpu_t, gpu_t, sp))
    print(f"{name:15s}  {cpu_t:>15.0f}  {gpu_t:>15.0f}  {sp:>7.1f}×")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

names_nn = [r[0] for r in nn_results]
x = np.arange(len(names_nn))
width = 0.35

ax1.bar(x - width/2, [r[1] for r in nn_results], width, label='CPU',
        color='#3498db', alpha=0.85)
ax1.bar(x + width/2, [r[2] for r in nn_results], width,
        label=str(device).upper(), color='#e74c3c', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(names_nn)
ax1.set_ylabel('Time per epoch (ms)')
ax1.set_title('Training Time per Epoch')
ax1.legend()

ax2.bar(names_nn, [r[3] for r in nn_results], color='#2ecc71', alpha=0.85)
ax2.axhline(y=1, color='gray', linestyle='--')
ax2.set_ylabel('Speedup')
ax2.set_title('GPU Speedup for Neural Network Training')
for i, r in enumerate(nn_results):
    ax2.text(i, r[3] + 0.2, f'{r[3]:.1f}×', ha='center', fontsize=10)

plt.suptitle('Neural Network Training: CPU vs GPU', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Batch Size Effect

GPUs execute thousands of operations in parallel. A **larger batch size** means more work per GPU call — better utilisation.  
Below we show how throughput (samples/second) changes with batch size on both devices.

In [ ]:
model_cpu = MLP(IN_DIM, 1024, OUT_DIM, 4)
model_gpu = MLP(IN_DIM, 1024, OUT_DIM, 4).to(device)

batch_sizes = [1, 8, 32, 64, 128, 256, 512, 1024, 2048]
cpu_throughput, gpu_throughput = [], []

print(f"{'Batch':>6s}  {'CPU (samp/s)':>13s}  {'GPU (samp/s)':>13s}  {'Speedup':>8s}")
print("-" * 48)

for bs in batch_sizes:
    xb_c = torch.randn(bs, IN_DIM)
    xb_g = xb_c.to(device)

    with torch.no_grad():
        c_ms, _ = benchmark(lambda: model_cpu(xb_c), n_runs=30)
        g_ms, _ = benchmark(lambda: model_gpu(xb_g), n_runs=30)

    c_sps = bs / (c_ms / 1000)
    g_sps = bs / (g_ms / 1000)
    cpu_throughput.append(c_sps)
    gpu_throughput.append(g_sps)
    print(f"{bs:>6d}  {c_sps:>13,.0f}  {g_sps:>13,.0f}  {g_sps/c_sps:>7.1f}×")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, [t/1000 for t in cpu_throughput], 'o-',
        label='CPU', color='#3498db', linewidth=2)
ax.plot(batch_sizes, [t/1000 for t in gpu_throughput], 's-',
        label=str(device).upper(), color='#e74c3c', linewidth=2)
ax.set_xlabel('Batch size')
ax.set_ylabel('Throughput (K samples/sec)')
ax.set_title('Inference Throughput vs Batch Size')
ax.set_xscale('log', base=2)
ax.set_xticks(batch_sizes)
ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
ax.legend()
plt.tight_layout()
plt.show()
print("GPU throughput keeps rising with batch size — CPU plateaus much earlier.")

---
## 7. Precision: FP32 vs FP16 (Mixed Precision)

Modern GPUs have dedicated **Tensor Cores** that run FP16 operations ~2–8× faster than FP32.  
PyTorch's `torch.autocast` enables mixed precision automatically — it uses FP16 where safe and FP32 where needed.

In [ ]:
N = 2048
A = torch.randn(N, N).to(device)
B = torch.randn(N, N).to(device)

# FP32 (default)
fp32_ms, _ = benchmark(lambda: torch.matmul(A, B))

# FP16 explicit cast
A16 = A.half()
B16 = B.half()
fp16_ms, _ = benchmark(lambda: torch.matmul(A16, B16))

# torch.autocast (mixed precision — preferred approach)
if device.type == 'cuda':
    def amp_matmul():
        with torch.autocast(device_type='cuda'):
            return torch.matmul(A, B)
    amp_ms, _ = benchmark(amp_matmul)
else:
    amp_ms = fp32_ms   # no AMP on CPU

print(f"Matrix multiply ({N}×{N}) on {device}:")
print(f"  FP32 (default):       {fp32_ms:.2f} ms")
print(f"  FP16 explicit:        {fp16_ms:.2f} ms  ({fp32_ms/fp16_ms:.1f}× vs FP32)")
print(f"  torch.autocast (AMP): {amp_ms:.2f} ms  ({fp32_ms/amp_ms:.1f}× vs FP32)")

if device.type == 'cuda':
    print(f"\nMemory: FP32={A.element_size()*A.numel()/1e6:.1f}MB  "
          f"FP16={A16.element_size()*A16.numel()/1e6:.1f}MB  (2× less memory)")
    print("Tip: use torch.autocast in training loops — it handles precision automatically.")
else:
    print("\n(AMP speedup only available on CUDA GPUs with Tensor Cores)")

---
## 8. GPU Memory Management

In [ ]:
def mem_stats(label):
    if device.type != 'cuda':
        ram = psutil.virtual_memory()
        print(f"{label:30s}  RAM used: {ram.used/1024**3:.2f} GB  ({ram.percent:.1f}%)")
        return
    alloc = torch.cuda.memory_allocated() / 1024**2
    reserv = torch.cuda.memory_reserved()  / 1024**2
    print(f"{label:30s}  allocated={alloc:7.1f} MB  reserved={reserv:7.1f} MB")


torch.cuda.empty_cache() if device.type == 'cuda' else None
mem_stats('Baseline')

# Allocate a large tensor
big = torch.randn(1024, 1024, 256, device=device)   # 1024*1024*256 * 4B ≈ 1 GB
mem_stats('After 1GB allocation')

# Delete Python reference — tensor is not freed until cache is cleared
del big
mem_stats('After del (cache not cleared)')

if device.type == 'cuda':
    torch.cuda.empty_cache()
    mem_stats('After empty_cache()')

print()
print("Key points:")
print("  • del only removes the Python reference — PyTorch caches the memory for reuse")
print("  • torch.cuda.empty_cache() releases cached memory back to the OS")
print("  • Call empty_cache() if you get OOM errors after freeing tensors")

---
## 9. When to Use GPU — Decision Guide

| Situation | Use GPU? | Reason |
|---|---|---|
| Large matrix multiply (N > 512) | **Yes** | GPU is purpose-built for this |
| Neural network training (medium/large) | **Yes** | Every forward/backward pass is matmuls |
| Batch size ≥ 64 | **Yes** | Enough parallelism to saturate GPU |
| Single inference, batch=1 | **Maybe** | Transfer overhead may dominate |
| Small tensors (N < 128) | **No** | CPU is faster — launch overhead is too high |
| Preprocessing / data loading | **No** | CPU-bound I/O; keep on CPU |
| FP16 / mixed precision | **Yes** | 2–8× additional speedup on modern GPUs |
| Repeated small kernel calls | **No** | Each CUDA launch has fixed overhead |

### The golden rule

```python
# Good: move data ONCE, do all computation on GPU
X = X.to(device)
for epoch in range(100):
    loss = model(X)         # stays on GPU each iteration

# Bad: move data every iteration
for epoch in range(100):
    X_gpu = X.to(device)    # transfer cost × 100 epochs
    loss = model(X_gpu)
    X_gpu.cpu()             # unnecessary round-trip
```

In [ ]:
# Final summary chart: all speedups in one view
all_labels, all_speedups, all_colors = [], [], []

# Matmul at largest size
all_labels.append(f'Matmul {sizes[-1]}×{sizes[-1]}')
all_speedups.append(speedups[-1])
all_colors.append('#3498db')

# Operations
for label, _, _, sp in op_results:
    all_labels.append(label)
    all_speedups.append(sp)
    all_colors.append('#9b59b6')

# Neural networks
for name, _, _, sp in nn_results:
    all_labels.append(name + ' (train)')
    all_speedups.append(sp)
    all_colors.append('#e74c3c')

# Sort
order = np.argsort(all_speedups)
all_labels   = [all_labels[i]   for i in order]
all_speedups = [all_speedups[i] for i in order]
all_colors   = [all_colors[i]   for i in order]

fig, ax = plt.subplots(figsize=(11, max(6, len(all_labels) * 0.4)))
bars = ax.barh(all_labels, all_speedups, color=all_colors, alpha=0.85)
ax.axvline(x=1, color='black', linestyle='--', linewidth=1, label='No speedup')
ax.set_xlabel(f'Speedup over CPU  (device: {device})')
ax.set_title('GPU Speedup Summary — All Operations')
for bar, sp in zip(bars, all_speedups):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{sp:.1f}×', va='center', fontsize=8)

from matplotlib.patches import Patch
legend = [
    Patch(color='#3498db', label='Matrix multiply'),
    Patch(color='#9b59b6', label='Element-wise ops'),
    Patch(color='#e74c3c', label='NN training'),
]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout()
plt.show()